In [2]:
%load_ext autoreload
%autoreload 2
import os
import numpy as np
from IPython.display import display
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torchvision import transforms
from torch.utils.data import DataLoader
import wandb
import h5py

import sys
sys.path.append('../../../src/benchmark/')
sys.path.append('../../../src/utils/')
from build_model import resnet50_
from train_functions import train_epochs
from utils import split_train_valid, list_to_dict, viz_dataloader, hdf5_dataset
from viz import show_images

In [1]:
symmetry_classes = ['p1', 'p2', 'pm', 'pg', 'cm', 'pmm', 'pmg', 'pgg', 'cmm', 'p4', 'p4m', 'p4g', 'p3', 'p3m1', 'p31m', 'p6', 'p6m']
label_converter = list_to_dict(symmetry_classes)

# imagenet
imagenet_ds = hdf5_dataset('../../../../imagenet_v5_rot_10m.h5', folder='train', transform=transforms.ToTensor())
_, imagenet_ds = split_train_valid(imagenet_ds, 0.9)
train_ds, valid_ds = split_train_valid(imagenet_ds, 0.8) # use total of 1m images
train_dl = DataLoader(train_ds, batch_size=800, shuffle=True, num_workers=4)
viz_dataloader(train_dl, label_converter=label_converter, title='imagenet - train')
valid_dl = DataLoader(valid_ds, batch_size=800, shuffle=False, num_workers=4)
viz_dataloader(valid_dl, label_converter=label_converter, title='imagenet - valid')

# atom
atom_ds = hdf5_dataset('../../../../atom_v5_rot_200k.h5', folder='test', transform=transforms.ToTensor())
atom_dl = DataLoader(atom_ds, batch_size=800, shuffle=False, num_workers=4)
viz_dataloader(atom_dl, label_converter=label_converter, title='atom - cv')

NameError: name 'list_to_dict' is not defined

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from e2cnn import gspaces
from e2cnn import nn as enn

# Dataset Preparation
transform = transforms.Compose([
    transforms.RandomRotation(180),  # Apply random rotations
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# Load and augment CIFAR-10 dataset
train_dataset = datasets.CIFAR10(root='./data', train=True, transform=transform, download=True)
test_dataset = datasets.CIFAR10(root='./data', train=False, transform=transform)

# Regular CNN Model
class RegularCNN(nn.Module):
    def __init__(self):
        super(RegularCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = F.relu(F.max_pool2d(self.conv1(x), 2))
        x = F.relu(F.max_pool2d(self.conv2(x), 2))
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# SE2CNN Model
class SE2CNN(nn.Module):
    def __init__(self, max_freq=5):
        super(SE2CNN, self).__init__()
        self.r2_act = gspaces.Rot2dOnR2(N=-1, maximum_frequency=max_freq)  # Continuous group of rotations
        self.input_type = enn.FieldType(self.r2_act, 1 * [self.r2_act.trivial_repr])
        
        # Use trivial_repr for the input and select appropriate representations for subsequent layers
        self.conv1 = enn.R2Conv(self.input_type, enn.FieldType(self.r2_act, 32 * [self.r2_act.trivial_repr]), kernel_size=3, padding=1)
        self.bn1 = enn.InnerBatchNorm(self.conv1.out_type)
        self.relu1 = enn.ReLU(self.conv1.out_type)
        
        self.conv2 = enn.R2Conv(self.conv1.out_type, enn.FieldType(self.r2_act, 64 * [self.r2_act.trivial_repr]), kernel_size=3, padding=1)
        self.bn2 = enn.InnerBatchNorm(self.conv2.out_type)
        self.relu2 = enn.ReLU(self.conv2.out_type)
        
        self.gpool = enn.GroupPooling(self.conv2.out_type)
        
        self.fc1 = nn.Linear(64 * 1 * 1, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = enn.GeometricTensor(x, self.input_type)
        x = self.relu1(self.bn1(self.conv1(x)))
        x = self.relu2(self.bn2(self.conv2(x)))
        x = self.gpool(x)
        x = x.tensor
        
        x = F.adaptive_avg_pool2d(x, 1).view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x
    
# Training Function
def train(model, train_loader, optimizer, criterion, device):
    model.train()
    for data, target in train_loader:
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()

# Testing Function
def test(model, test_loader, criterion, device):
    model.eval()
    test_loss = 0
    correct = 0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss += criterion(output, target).item()
            pred = output.argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum().item()

    test_loss /= len(test_loader.dataset)
    accuracy = correct / len(test_loader.dataset)
    return test_loss, accuracy

# Main Experiment
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
num_epochs = 10
criterion = nn.CrossEntropyLoss()

# Regular CNN
regular_model = RegularCNN().to(device)
regular_optimizer = torch.optim.Adam(regular_model.parameters(), lr=0.001)

# SE2CNN
se2_model = SE2CNN().to(device)
se2_optimizer = torch.optim.Adam(se2_model.parameters(), lr=0.001)

# Training Loop
for epoch in range(num_epochs):
    train(regular_model, train_loader, regular_optimizer, criterion, device)
    train(se2_model, train_loader, se2_optimizer, criterion, device)
    
    regular_loss, regular_acc = test(regular_model, test_loader, criterion, device)
    se2_loss, se2_acc = test(se2_model, test_loader, criterion, device)
    
    print(f'Epoch {epoch+1}/{num_epochs}')
    print(f'Regular CNN - Loss: {regular_loss:.4f}, Accuracy: {regular_acc:.4f}')
    print(f'SE2CNN      - Loss: {se2_loss:.4f}, Accuracy: {se2_acc:.4f}')
     

100%|██████████| 170498071/170498071 [00:02<00:00, 64966450.83it/s] 


Extracting ./data/cifar-10-python.tar.gz to ./data
Epoch 1/10
Regular CNN - Loss: 0.0048, Accuracy: 0.8998
SE2CNN      - Loss: 0.0318, Accuracy: 0.2290
Epoch 2/10
Regular CNN - Loss: 0.0034, Accuracy: 0.9311
SE2CNN      - Loss: 0.0280, Accuracy: 0.3383
Epoch 3/10
Regular CNN - Loss: 0.0029, Accuracy: 0.9406
SE2CNN      - Loss: 0.0446, Accuracy: 0.2565
Epoch 4/10
Regular CNN - Loss: 0.0024, Accuracy: 0.9534
SE2CNN      - Loss: 0.0479, Accuracy: 0.2891
Epoch 5/10
Regular CNN - Loss: 0.0020, Accuracy: 0.9586
SE2CNN      - Loss: 0.0507, Accuracy: 0.1493
Epoch 6/10
Regular CNN - Loss: 0.0020, Accuracy: 0.9591
SE2CNN      - Loss: 0.0213, Accuracy: 0.5147
Epoch 7/10
Regular CNN - Loss: 0.0022, Accuracy: 0.9556
SE2CNN      - Loss: 0.0305, Accuracy: 0.3824
Epoch 8/10
Regular CNN - Loss: 0.0017, Accuracy: 0.9646
SE2CNN      - Loss: 0.0207, Accuracy: 0.5327
Epoch 9/10
Regular CNN - Loss: 0.0015, Accuracy: 0.9691
SE2CNN      - Loss: 0.0208, Accuracy: 0.5139
Epoch 10/10
Regular CNN - Loss: 0.0016, 

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from e2cnn import gspaces
from e2cnn import nn as enn

# Dataset Preparation
transform = transforms.Compose([
    transforms.RandomRotation(180),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# Load and augment MNIST dataset
train_dataset = datasets.MNIST(root='./data', train=True, transform=transform, download=True)
test_dataset = datasets.MNIST(root='./data', train=False, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# Regular CNN Model
class RegularCNN(nn.Module):
    def __init__(self):
        super(RegularCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = F.relu(F.max_pool2d(self.conv1(x), 2))
        x = F.relu(F.max_pool2d(self.conv2(x), 2))
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x


# SE2CNN Model
class SE2CNN(nn.Module):
    def __init__(self, max_freq=5):
        super(SE2CNN, self).__init__()
        self.r2_act = gspaces.Rot2dOnR2(N=-1, maximum_frequency=max_freq)  # Continuous group of rotations
        self.input_type = enn.FieldType(self.r2_act, 1 * [self.r2_act.trivial_repr])
        
        # Use trivial_repr for the input and select appropriate representations for subsequent layers
        self.conv1 = enn.R2Conv(self.input_type, enn.FieldType(self.r2_act, 32 * [self.r2_act.trivial_repr]), kernel_size=3, padding=1)
        self.bn1 = enn.InnerBatchNorm(self.conv1.out_type)
        self.relu1 = enn.ReLU(self.conv1.out_type)
        
        self.conv2 = enn.R2Conv(self.conv1.out_type, enn.FieldType(self.r2_act, 64 * [self.r2_act.trivial_repr]), kernel_size=3, padding=1)
        self.bn2 = enn.InnerBatchNorm(self.conv2.out_type)
        self.relu2 = enn.ReLU(self.conv2.out_type)
        
        self.gpool = enn.GroupPooling(self.conv2.out_type)
        
        self.fc1 = nn.Linear(64, 32)
        self.fc2 = nn.Linear(32, 10)

    def forward(self, x):
        x = enn.GeometricTensor(x, self.input_type)
        x = self.relu1(self.bn1(self.conv1(x)))
        x = self.relu2(self.bn2(self.conv2(x)))
        x = self.gpool(x)
        x = x.tensor
        
        x = F.adaptive_avg_pool2d(x, 1).view(x.size(0), -1)
        # print(x.size())
        x = self.fc1(x)
        # print(x.size())
        x = F.relu(x)
        # print(x.size())
        x = self.fc2(x)
        return x
    
# Training Function
def train(model, train_loader, optimizer, criterion, device):
    model.train()
    for data, target in train_loader:
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()

# Testing Function
def test(model, test_loader, criterion, device):
    model.eval()
    test_loss = 0
    correct = 0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss += criterion(output, target).item()
            pred = output.argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum().item()

    test_loss /= len(test_loader.dataset)
    accuracy = correct / len(test_loader.dataset)
    return test_loss, accuracy

# Main Experiment
device = torch.device('cuda:7' if torch.cuda.is_available() else 'cpu')
num_epochs = 10
criterion = nn.CrossEntropyLoss()

# Regular CNN
regular_model = RegularCNN().to(device)
regular_optimizer = torch.optim.Adam(regular_model.parameters(), lr=0.001)

# SE2CNN
se2_model = SE2CNN().to(device)
se2_optimizer = torch.optim.Adam(se2_model.parameters(), lr=0.001)

print(se2_model)
print(regular_model)

SE2CNN(
  (conv1): R2Conv([Continuous-Rotations: {irrep_0}], [Continuous-Rotations: {irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0}], kernel_size=3, stride=1, padding=1)
  (bn1): InnerBatchNorm([Continuous-Rotations: {irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0}], eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu1): ReLU(inplace=False, type=[Continuous-Rotations: {irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, irrep_0, i

In [3]:
# Training Loop
for epoch in range(100):
    train(regular_model, train_loader, regular_optimizer, criterion, device)
    train(se2_model, train_loader, se2_optimizer, criterion, device)
    
    regular_loss, regular_acc = test(regular_model, test_loader, criterion, device)
    se2_loss, se2_acc = test(se2_model, test_loader, criterion, device)
    
    print(f'Epoch {epoch+1}/{num_epochs}')
    print(f'Regular CNN - Loss: {regular_loss:.4f}, Accuracy: {regular_acc:.4f}')
    print(f'SE2CNN      - Loss: {se2_loss:.4f}, Accuracy: {se2_acc:.4f}')

Epoch 1/10
Regular CNN - Loss: 0.0015, Accuracy: 0.9689
SE2CNN      - Loss: 0.0208, Accuracy: 0.5255
Epoch 2/10
Regular CNN - Loss: 0.0015, Accuracy: 0.9711
SE2CNN      - Loss: 0.0199, Accuracy: 0.5590
Epoch 3/10
Regular CNN - Loss: 0.0016, Accuracy: 0.9670
SE2CNN      - Loss: 0.0273, Accuracy: 0.4166
Epoch 4/10
Regular CNN - Loss: 0.0015, Accuracy: 0.9716
SE2CNN      - Loss: 0.0206, Accuracy: 0.5189
Epoch 5/10
Regular CNN - Loss: 0.0015, Accuracy: 0.9710
SE2CNN      - Loss: 0.0199, Accuracy: 0.5546
Epoch 6/10
Regular CNN - Loss: 0.0013, Accuracy: 0.9757
SE2CNN      - Loss: 0.0341, Accuracy: 0.3061
Epoch 7/10
Regular CNN - Loss: 0.0013, Accuracy: 0.9730
SE2CNN      - Loss: 0.0312, Accuracy: 0.3676
Epoch 8/10
Regular CNN - Loss: 0.0014, Accuracy: 0.9727
SE2CNN      - Loss: 0.0256, Accuracy: 0.4178
Epoch 9/10
Regular CNN - Loss: 0.0014, Accuracy: 0.9725
SE2CNN      - Loss: 0.0277, Accuracy: 0.3924
Epoch 10/10
Regular CNN - Loss: 0.0013, Accuracy: 0.9739
SE2CNN      - Loss: 0.0698, Accura

In [ ]:
# noise
noise_ds = hdf5_dataset('../../../../imagenet_atom_noise_v4_rot_10m_100k_subset.h5', folder='noise', transform=transforms.ToTensor())
noise_dl = DataLoader(noise_ds, batch_size=800, shuffle=False, num_workers=4)
viz_dataloader(noise_dl, label_converter=label_converter, title='noise - cv')

In [4]:
model = resnet50_(in_channels=3, n_classes=17)
outputs = model(torch.randn(2,3,256,256))
print(outputs.shape)
model = torch.nn.DataParallel(model, device_ids=[1,3])
# model

torch.Size([2, 17])


In [5]:
config = {
    'dataset': '10 million datasets',
    'loss_func': 'CrossEntropyLoss', # nn.MSELoss()
    'optimizer': 'Adam',
    'scheduler': 'OneCycleLR',
}

NAME = '05242024-benchmark-resnet50_from_scratch-v5_10m'

# train

In [6]:
wandb.login()
proj_name = 'Understanding-Experimental-Images-by-Identifying-Symmetries-with-Deep-Learning'
wandb.init(project=proj_name, entity='yig319', name=NAME, id=NAME, save_code=True, config=config)
config = wandb.config

Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Currently logged in as: yig319. Use `wandb login --relogin` to force relogin


In [7]:
device = torch.device('cuda:1')
lr = 1e-3
start = 0
epochs = 20

loss_func = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, epochs=epochs, max_lr=lr, steps_per_epoch=len(train_dl))
history = train_epochs(model, loss_func, optimizer, device, train_dl, valid_dl, cv_dl_list=[atom_dl], cv_name_list=['atom'],
                       epochs=epochs, start=start, scheduler=scheduler, model_dir='../../../saved_models/', tracking=False)

Epoch: 1/20


  0%|          | 0/10711 [00:00<?, ?it/s]

100%|██████████| 10711/10711 [3:12:04<00:00,  1.08s/it] 


Training: Loss: 0.9434, Accuracy: 68.3690%.


100%|██████████| 2678/2678 [32:34<00:00,  1.37it/s]


...

Training: Loss: 0.0357, Accuracy: 98.7655%.


100%|██████████| 2678/2678 [52:28<00:00,  1.18s/it] 


Validation: Loss: 0.0474, Accuracy: 98.6072%.


100%|██████████| 255/255 [04:05<00:00,  1.04it/s]


atom: Loss: 1.5499, Accuracy: 84.0907%.


In [8]:
device = torch.device('cuda:1')
lr = 1e-3
start = 20
epochs = 20

loss_func = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, epochs=epochs, max_lr=lr, steps_per_epoch=len(train_dl))
history = train_epochs(model, loss_func, optimizer, device, train_dl, valid_dl, cv_dl_list=[atom_dl], cv_name_list=['atom'],
                       epochs=epochs, start=start, scheduler=scheduler, model_dir='../../../saved_models/', tracking=False)

Epoch: 21/40


100%|██████████| 10711/10711 [3:25:02<00:00,  1.15s/it] 


Training: Loss: 0.0358, Accuracy: 98.7619%.


100%|██████████| 2678/2678 [38:50<00:00,  1.15it/s] 


Validation: Loss: 0.0487, Accuracy: 98.6004%.


...

100%|██████████| 2678/2678 [42:15<00:00,  1.06it/s] 


Validation: Loss: 0.0494, Accuracy: 98.5107%.


100%|██████████| 255/255 [04:37<00:00,  1.09s/it]


atom: Loss: 1.6326, Accuracy: 78.2637%.
Epoch: 29/40


 51%|█████▏    | 5500/10711 [1:44:00<1:34:42,  1.09s/it]